In [ ]:
import numpy as np 
import pypulseq as pq 
import matplotlib.pyplot as plt 

# maximum gradient amplitude [mT/m]
max_grad = 40.0

# maximum gradient slew rate [T/m/s]
# set super low here to eliminate PNS risks 
max_slew = 90.0

# duration of each sample in RF pulse [s]
rf_raster_time = 2.0e-6 

# necessary post-rf delay [s]
rf_ringdown_time = 60.0e-6

# necessary pre-rf delay [s]
rf_dead_time = 100.0e-6 

# necessary pre-adc delay [s] 
adc_dead_time = 40.0e-6 

# adc raster time [s]
adc_raster_time = 2.0e-6 

# gradient raster time [s]
grad_raster_time = 4.0e-6 

# total duration of each block must be evenly divisible by this number
block_duration_raster = 4.0e-6 

# delay inserted between segments for GE systems 
# added by GE pge2 interpreter 
end_of_segment_delay = 116.0e-6

# make the system object for PyPulseq 
system = pq.Opts(max_grad=max_grad,
                     grad_unit='mT/m',
                     max_slew=max_slew,
                     slew_unit='T/m/s',
                     rf_ringdown_time=rf_ringdown_time,
                     rf_dead_time=rf_dead_time,
                     rf_raster_time=rf_raster_time,
                     adc_dead_time=adc_dead_time,
                     adc_raster_time=adc_raster_time,
                     grad_raster_time=grad_raster_time,
                     block_duration_raster=block_duration_raster)

fovx = 0.22 # frequency encoding FOV [m]
fovy = 0.22 # phase encoding FOV [m]
dz = 0.005  # slice thickness [m]

# imaging matrix sizes 
nx = 128 
ny = 128 

# TODO: choose inversion times 
TI = np.array([], dtype=np.float32)

# TODO: choose this
TR = 5.0 # repetition time [s]

# TODO: choose this
flip_angle = 1.0 
alpha = flip_angle * np.pi / 180 

# TODO: choose this (make a multiple of 2 microseconds)
adc_dwell_time = 8.0e-6

# TODO: calculate frequency encoding gradient amplitude in units of Hz/m 
Gx_Hz_m = 0.0 

# TODO: calculate duration of flat part of frequency encoding gradient 
Gx_flat_time = 0.01

# make the frequency encoding gradient 
Gx = pq.make_trapezoid(channel='x', amplitude=Gx_Hz_m, flat_time=Gx_flat_time, system=system)

# TODO: make the frequency encoding prephaser 
area_Gx_pre = 0.0 # calculate this! 
Gx_pre = pq.make_trapezoid(channel='x', area=area_Gx_pre, system=system)

# TODO: calculate phase encoding areas (units of 1/m)
delta_ky = 0.0
phase_areas = None # you make this, it should be a numpy array with length of ny
max_phase_area = np.max(np.abs(phase_areas))
phase_scales = phase_areas / max_phase_area # factor by which to scale amplitude of Gy gradient for each phase encoding step

# make the phase encoding gradient 
Gy = pq.make_trapezoid(channel='y', area=max_phase_area, system=system)

# TODO: make the inversion pulse 
inv_duration = 1.0e-3 
inv_flip_angle = 0.0 # TODO: set this
rf_inv = pq.make_block_pulse(flip_angle=inv_flip_angle, duration=inv_duration, system=system)

# TODO: make the slice-selective excitation pulse 
exc_bw = 0.0 # TODO: choose this
exc_duration = 0.0 # TODO: choose this 
exc_tbw = exc_bw * exc_duration 
rf_exc, gz = pq.make_sinc_pulse(flip_angle=alpha, duration=exc_duration, time_bw_product=exc_tbw, slice_thickness=dz, return_gz=True, system=system)


